# ⚡ Fast Movement Anomaly Detection

Notebook ini mendeteksi **anomali pergerakan cepat** pada data GPS:

1. **Kecepatan tidak wajar** — seseorang berpindah lokasi lebih cepat dari batas fisika manusia
   (misal > 300 km/jam, melebihi kecepatan kereta cepat)
2. **Lokasi simultan (bi-location)** — MAID yang sama tercatat di dua lokasi berbeda
   pada waktu yang sama persis (timestamp identik)

> **Pendekatan: DuckDB dengan Disk Spilling + Batch Processing**
>
> Data dibagi ke dalam **20 hash bucket** berdasarkan MAID, lalu setiap bucket
> diproses terpisah dengan window function. Ini memastikan setiap batch hanya
> ~15 juta baris sehingga tidak OOM pada 3.7 GB RAM.

**Output:**
- Statistik jumlah anomali per jenis
- Distribusi kecepatan anomali
- Export record anomali ke Parquet

---
## 1 · Setup & Konfigurasi

In [1]:
import os
import time

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from shared import get_con, setup_matplotlib

# ── Konfigurasi ──────────────────────────────────────────────────────────────
BASE_DIR = os.getcwd()
INPUT_FILE = os.path.join(BASE_DIR, "DataGPS_parquet", "gps_ring_road_utara.parquet")
OUTPUT_DIR = os.path.join(BASE_DIR, "fast-movement-anomali-ring-road")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Threshold kecepatan (km/jam)
# 150 km/jam ≈ kendaraan darat
SPEED_THRESHOLD_KMPH = 150

# Jarak minimum untuk dipertimbangkan (meter)
# Ping yang berjarak < 50m diabaikan karena noise GPS
MIN_DISTANCE_M = 50

# Jumlah bucket untuk batch processing (menghindari OOM)
NUM_BUCKETS = 20

# DuckDB connection dengan disk spilling
con = get_con(BASE_DIR)
setup_matplotlib()

print("⚙️  DuckDB Configuration:")
for setting in ["memory_limit", "threads", "temp_directory"]:
    val = con.execute(f"SELECT current_setting('{setting}')").fetchone()[0]
    print(f"   {setting}: {val}")

print(f"\n📁 Input  : {INPUT_FILE}")
print(f"📁 Output : {OUTPUT_DIR}/")
print(f"🚀 Speed threshold : {SPEED_THRESHOLD_KMPH} km/jam")
print(f"📏 Min distance    : {MIN_DISTANCE_M} m")
print(f"🪣 Batch buckets   : {NUM_BUCKETS}")

⚙️  DuckDB Configuration:
   memory_limit: 3.7 GiB
   threads: 4
   temp_directory: /home/repal/Github/skripsi/./.duckdb_tmp

📁 Input  : /home/repal/Github/skripsi/DataGPS_parquet/gps_ring_road_utara.parquet
📁 Output : /home/repal/Github/skripsi/fast-movement-anomali-ring-road/
🚀 Speed threshold : 150 km/jam
📏 Min distance    : 50 m
🪣 Batch buckets   : 20


---
## 2 · Verifikasi Data Input

In [2]:
assert os.path.exists(INPUT_FILE), f"❌ File tidak ditemukan: {INPUT_FILE}"

file_size_mb = os.path.getsize(INPUT_FILE) / 1e6
total_rows = con.execute(
    "SELECT COUNT(*) FROM read_parquet($1)", [INPUT_FILE]
).fetchone()[0]

distinct_maid = con.execute(
    "SELECT COUNT(DISTINCT maid) FROM read_parquet($1)", [INPUT_FILE]
).fetchone()[0]

print("=" * 70)
print("VERIFIKASI DATA INPUT")
print("=" * 70)
print(f"  📁 File           : {os.path.basename(INPUT_FILE)}")
print(f"  📏 Ukuran         : {file_size_mb:.1f} MB")
print(f"  📊 Total baris    : {total_rows:,}")
print(f"  📍 Distinct MAID  : {distinct_maid:,}")
print(f"  ✅ Data siap diproses")

VERIFIKASI DATA INPUT
  📁 File           : gps_ring_road_utara.parquet
  📏 Ukuran         : 64.1 MB
  📊 Total baris    : 8,963,571
  📍 Distinct MAID  : 400,643
  ✅ Data siap diproses


---
## 3 · Haversine Macro

In [3]:
# Definisikan macro Haversine sebagai DuckDB macro
# Mengembalikan jarak dalam METER antara dua titik lat/lon
con.execute("""
    CREATE OR REPLACE MACRO haversine_m(lat1, lon1, lat2, lon2) AS (
        2 * 6371000 * ASIN(
            SQRT(
                POWER(SIN(RADIANS(lat2 - lat1) / 2), 2)
                + COS(RADIANS(lat1)) * COS(RADIANS(lat2))
                  * POWER(SIN(RADIANS(lon2 - lon1) / 2), 2)
            )
        )
    )
""")

# Verifikasi: jarak Yogyakarta (-7.797, 110.361) → Jakarta (-6.200, 106.816)
test_dist = con.execute("""
    SELECT haversine_m(-7.797, 110.361, -6.200, 106.816) / 1000 AS dist_km
""").fetchone()[0]
print(f"✅ Haversine macro OK — Yogya→Jakarta = {test_dist:.1f} km (expected ~430 km)")

✅ Haversine macro OK — Yogya→Jakarta = 429.6 km (expected ~430 km)


---
## 4 · Deteksi Anomali Tipe 1: Kecepatan Tidak Wajar

Untuk setiap MAID, kita bandingkan setiap ping dengan ping **sebelumnya**
(berdasarkan urutan timestamp). Lalu hitung:

- `distance_m` = jarak Haversine antara 2 ping berurutan
- `time_diff_s` = selisih waktu dalam detik
- `speed_kmph` = (distance_m / time_diff_s) × 3.6

Jika `speed_kmph > SPEED_THRESHOLD_KMPH` → **anomali**.

**Strategi anti-OOM:** data dibagi ke 20 bucket berdasarkan `hash(maid) % 20`.
Setiap bucket diproses terpisah (~15 juta baris), lalu digabung.

In [4]:
print("=" * 70)
print("DETEKSI ANOMALI TIPE 1: KECEPATAN TIDAK WAJAR")
print(f"Threshold: > {SPEED_THRESHOLD_KMPH} km/jam, jarak minimum: {MIN_DISTANCE_M} m")
print("=" * 70)

SPEED_ANOMALY_FILE = os.path.join(OUTPUT_DIR, "speed_anomalies.parquet")
SPEED_BUCKET_DIR = os.path.join(OUTPUT_DIR, "_speed_buckets")
os.makedirs(SPEED_BUCKET_DIR, exist_ok=True)

# Kurangi threads untuk operasi berat agar hemat RAM
con.execute("SET threads=2")

t0 = time.time()
print(f"\n⏳ Memproses {NUM_BUCKETS} bucket MAID (window function per bucket)...")
print(f"   ~{total_rows // NUM_BUCKETS:,} baris/bucket\n")

bucket_files = []
for bucket_id in range(NUM_BUCKETS):
    bucket_file = os.path.join(SPEED_BUCKET_DIR, f"bucket_{bucket_id:02d}.parquet")
    bucket_files.append(bucket_file)

    con.execute(f"""
        COPY (
            WITH src AS (
                SELECT *
                FROM read_parquet('{INPUT_FILE}')
                WHERE hash(maid) % {NUM_BUCKETS} = {bucket_id}
            ),
            lagged AS (
                SELECT
                    maid,
                    latitude,
                    longitude,
                    timestamp,
                    LAG(latitude)  OVER w AS prev_lat,
                    LAG(longitude) OVER w AS prev_lon,
                    LAG(timestamp) OVER w AS prev_ts
                FROM src
                WINDOW w AS (PARTITION BY maid ORDER BY timestamp)
            ),
            computed AS (
                SELECT
                    maid,
                    prev_lat,
                    prev_lon,
                    TO_TIMESTAMP(prev_ts) AS prev_time,
                    latitude  AS curr_lat,
                    longitude AS curr_lon,
                    TO_TIMESTAMP(timestamp) AS curr_time,
                    (timestamp - prev_ts)  AS time_diff_s,
                    haversine_m(prev_lat, prev_lon, latitude, longitude) AS distance_m,
                    CASE
                        WHEN (timestamp - prev_ts) > 0
                        THEN (haversine_m(prev_lat, prev_lon, latitude, longitude)
                              / (timestamp - prev_ts)) * 3.6
                        ELSE NULL
                    END AS speed_kmph
                FROM lagged
                WHERE prev_ts IS NOT NULL
            )
            SELECT *
            FROM computed
            WHERE speed_kmph > {SPEED_THRESHOLD_KMPH}
              AND distance_m >= {MIN_DISTANCE_M}
        )
        TO '{bucket_file}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    elapsed_b = time.time() - t0
    print(f"  ✅ Bucket {bucket_id + 1:2d}/{NUM_BUCKETS} selesai ({elapsed_b:.0f}s)", flush=True)

# Gabungkan semua bucket menjadi 1 file
print(f"\n⏳ Menggabungkan {NUM_BUCKETS} bucket...")
bucket_glob = os.path.join(SPEED_BUCKET_DIR, "bucket_*.parquet")
con.execute(f"""
    COPY (
        SELECT * FROM read_parquet('{bucket_glob}')
    )
    TO '{SPEED_ANOMALY_FILE}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# Kembalikan threads
con.execute("SET threads=4")

elapsed = time.time() - t0
print(f"\n✅ Selesai dalam {elapsed:.1f} detik ({elapsed / 60:.1f} menit)")
print(f"📁 Output: {SPEED_ANOMALY_FILE}")
print(f"📏 Ukuran: {os.path.getsize(SPEED_ANOMALY_FILE) / 1e6:.1f} MB")

DETEKSI ANOMALI TIPE 1: KECEPATAN TIDAK WAJAR
Threshold: > 150 km/jam, jarak minimum: 50 m

⏳ Memproses 20 bucket MAID (window function per bucket)...
   ~448,178 baris/bucket



  ✅ Bucket  1/20 selesai (0s)
  ✅ Bucket  2/20 selesai (1s)
  ✅ Bucket  3/20 selesai (1s)
  ✅ Bucket  4/20 selesai (1s)
  ✅ Bucket  5/20 selesai (1s)
  ✅ Bucket  6/20 selesai (2s)
  ✅ Bucket  7/20 selesai (2s)
  ✅ Bucket  8/20 selesai (2s)
  ✅ Bucket  9/20 selesai (2s)
  ✅ Bucket 10/20 selesai (2s)
  ✅ Bucket 11/20 selesai (3s)
  ✅ Bucket 12/20 selesai (3s)
  ✅ Bucket 13/20 selesai (3s)
  ✅ Bucket 14/20 selesai (4s)
  ✅ Bucket 15/20 selesai (4s)
  ✅ Bucket 16/20 selesai (4s)
  ✅ Bucket 17/20 selesai (4s)
  ✅ Bucket 18/20 selesai (5s)
  ✅ Bucket 19/20 selesai (5s)
  ✅ Bucket 20/20 selesai (5s)

⏳ Menggabungkan 20 bucket...

✅ Selesai dalam 5.3 detik (0.1 menit)
📁 Output: /home/repal/Github/skripsi/fast-movement-anomali-ring-road/speed_anomalies.parquet
📏 Ukuran: 0.8 MB


In [5]:
# Statistik anomali kecepatan
speed_stats = con.execute(f"""
    SELECT
        COUNT(*)                     AS total_anomali,
        COUNT(DISTINCT maid)         AS distinct_maid,
        MIN(speed_kmph)              AS min_speed,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY speed_kmph) AS p25_speed,
        PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY speed_kmph) AS median_speed,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY speed_kmph) AS p75_speed,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY speed_kmph) AS p95_speed,
        MAX(speed_kmph)              AS max_speed,
        AVG(distance_m) / 1000       AS avg_distance_km,
        AVG(time_diff_s)             AS avg_time_diff_s
    FROM read_parquet('{SPEED_ANOMALY_FILE}')
""").df()

print("=" * 70)
print("STATISTIK ANOMALI KECEPATAN")
print("=" * 70)
print(f"  🔴 Total record anomali   : {speed_stats['total_anomali'].iloc[0]:,}")
print(f"  👤 MAID terlibat          : {speed_stats['distinct_maid'].iloc[0]:,}")
print(f"  📊 Kecepatan (km/jam):")
print(f"     Min    : {speed_stats['min_speed'].iloc[0]:,.1f}")
print(f"     P25    : {speed_stats['p25_speed'].iloc[0]:,.1f}")
print(f"     Median : {speed_stats['median_speed'].iloc[0]:,.1f}")
print(f"     P75    : {speed_stats['p75_speed'].iloc[0]:,.1f}")
print(f"     P95    : {speed_stats['p95_speed'].iloc[0]:,.1f}")
print(f"     Max    : {speed_stats['max_speed'].iloc[0]:,.1f}")
print(f"  📏 Rata-rata jarak        : {speed_stats['avg_distance_km'].iloc[0]:,.2f} km")
print(f"  ⏱️  Rata-rata Δt           : {speed_stats['avg_time_diff_s'].iloc[0]:,.1f} detik")

STATISTIK ANOMALI KECEPATAN
  🔴 Total record anomali   : 16,910
  👤 MAID terlibat          : 4,486
  📊 Kecepatan (km/jam):
     Min    : 150.0
     P25    : 184.9
     Median : 246.6
     P75    : 426.5
     P95    : 1,407.0
     Max    : 36,004.2
  📏 Rata-rata jarak        : 1.00 km
  ⏱️  Rata-rata Δt           : 12.2 detik


---
## 5 · Deteksi Anomali Tipe 2: Bi-Location (Lokasi Simultan)

Mendeteksi MAID yang memiliki **timestamp identik** namun **lokasi berbeda**.
Ini mengindikasikan data duplikat/spoof — satu device tidak bisa berada
di dua tempat sekaligus.

Kriteria:
- Timestamp yang sama persis
- Jarak antar lokasi ≥ 100 meter (bukan noise GPS)

Sama seperti Tipe 1, diproses per bucket untuk menghindari OOM.

In [6]:
print("=" * 70)
print("DETEKSI ANOMALI TIPE 2: BI-LOCATION (LOKASI SIMULTAN)")
print("=" * 70)

BI_LOCATION_MIN_DIST_M = 100  # minimal 100m beda lokasi

BILOC_ANOMALY_FILE = os.path.join(OUTPUT_DIR, "bilocation_anomalies.parquet")
BILOC_BUCKET_DIR = os.path.join(OUTPUT_DIR, "_biloc_buckets")
os.makedirs(BILOC_BUCKET_DIR, exist_ok=True)

con.execute("SET threads=2")

t0 = time.time()
print(f"\n⏳ Mencari MAID dengan timestamp identik di lokasi berbeda (≥{BI_LOCATION_MIN_DIST_M}m)...")
print(f"   Memproses {NUM_BUCKETS} bucket...\n")

biloc_bucket_files = []
for bucket_id in range(NUM_BUCKETS):
    bucket_file = os.path.join(BILOC_BUCKET_DIR, f"bucket_{bucket_id:02d}.parquet")
    biloc_bucket_files.append(bucket_file)

    con.execute(f"""
        COPY (
            WITH src AS (
                SELECT maid, latitude, longitude, timestamp
                FROM read_parquet('{INPUT_FILE}')
                WHERE hash(maid) % {NUM_BUCKETS} = {bucket_id}
            ),
            dup_ts AS (
                SELECT maid, timestamp
                FROM src
                GROUP BY maid, timestamp
                HAVING COUNT(*) > 1
            ),
            dup_records AS (
                SELECT g.maid, g.latitude, g.longitude, g.timestamp
                FROM src g
                SEMI JOIN dup_ts d
                    ON g.maid = d.maid AND g.timestamp = d.timestamp
            ),
            paired AS (
                SELECT
                    a.maid,
                    TO_TIMESTAMP(a.timestamp) AS event_time,
                    a.latitude  AS lat_a,
                    a.longitude AS lon_a,
                    b.latitude  AS lat_b,
                    b.longitude AS lon_b,
                    haversine_m(a.latitude, a.longitude,
                                b.latitude, b.longitude) AS distance_m
                FROM dup_records a
                JOIN dup_records b
                    ON a.maid = b.maid
                    AND a.timestamp = b.timestamp
                    AND (a.latitude < b.latitude
                         OR (a.latitude = b.latitude AND a.longitude < b.longitude))
            )
            SELECT *
            FROM paired
            WHERE distance_m >= {BI_LOCATION_MIN_DIST_M}
        )
        TO '{bucket_file}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    elapsed_b = time.time() - t0
    print(f"  ✅ Bucket {bucket_id + 1:2d}/{NUM_BUCKETS} selesai ({elapsed_b:.0f}s)", flush=True)

# Gabungkan semua bucket
print(f"\n⏳ Menggabungkan {NUM_BUCKETS} bucket...")
biloc_glob = os.path.join(BILOC_BUCKET_DIR, "bucket_*.parquet")
con.execute(f"""
    COPY (
        SELECT * FROM read_parquet('{biloc_glob}')
    )
    TO '{BILOC_ANOMALY_FILE}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.execute("SET threads=4")

elapsed = time.time() - t0
print(f"\n✅ Selesai dalam {elapsed:.1f} detik ({elapsed / 60:.1f} menit)")
print(f"📁 Output: {BILOC_ANOMALY_FILE}")
print(f"📏 Ukuran: {os.path.getsize(BILOC_ANOMALY_FILE) / 1e6:.1f} MB")

DETEKSI ANOMALI TIPE 2: BI-LOCATION (LOKASI SIMULTAN)

⏳ Mencari MAID dengan timestamp identik di lokasi berbeda (≥100m)...
   Memproses 20 bucket...

  ✅ Bucket  1/20 selesai (0s)
  ✅ Bucket  2/20 selesai (1s)
  ✅ Bucket  3/20 selesai (1s)
  ✅ Bucket  4/20 selesai (1s)
  ✅ Bucket  5/20 selesai (2s)
  ✅ Bucket  6/20 selesai (2s)
  ✅ Bucket  7/20 selesai (2s)
  ✅ Bucket  8/20 selesai (2s)
  ✅ Bucket  9/20 selesai (3s)
  ✅ Bucket 10/20 selesai (3s)
  ✅ Bucket 11/20 selesai (3s)
  ✅ Bucket 12/20 selesai (3s)
  ✅ Bucket 13/20 selesai (4s)
  ✅ Bucket 14/20 selesai (4s)
  ✅ Bucket 15/20 selesai (4s)
  ✅ Bucket 16/20 selesai (4s)
  ✅ Bucket 17/20 selesai (5s)
  ✅ Bucket 18/20 selesai (5s)
  ✅ Bucket 19/20 selesai (5s)
  ✅ Bucket 20/20 selesai (5s)

⏳ Menggabungkan 20 bucket...

✅ Selesai dalam 5.4 detik (0.1 menit)
📁 Output: /home/repal/Github/skripsi/fast-movement-anomali-ring-road/bilocation_anomalies.parquet
📏 Ukuran: 0.0 MB


In [7]:
# Statistik anomali bi-location
biloc_stats = con.execute(f"""
    SELECT
        COUNT(*)                     AS total_pairs,
        COUNT(DISTINCT maid)         AS distinct_maid,
        MIN(distance_m)              AS min_dist_m,
        PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY distance_m) AS median_dist_m,
        MAX(distance_m)              AS max_dist_m,
        AVG(distance_m) / 1000       AS avg_dist_km
    FROM read_parquet('{BILOC_ANOMALY_FILE}')
""").df()

print("=" * 70)
print("STATISTIK ANOMALI BI-LOCATION")
print("=" * 70)
print(f"  🔴 Total pasangan anomali : {biloc_stats['total_pairs'].iloc[0]:,}")
print(f"  👤 MAID terlibat          : {biloc_stats['distinct_maid'].iloc[0]:,}")
print(f"  📏 Jarak antar lokasi simultan:")
print(f"     Min    : {biloc_stats['min_dist_m'].iloc[0]:,.1f} m")
print(f"     Median : {biloc_stats['median_dist_m'].iloc[0]:,.1f} m")
print(f"     Max    : {biloc_stats['max_dist_m'].iloc[0]:,.1f} m")
print(f"     Avg    : {biloc_stats['avg_dist_km'].iloc[0]:,.2f} km")

STATISTIK ANOMALI BI-LOCATION
  🔴 Total pasangan anomali : 1,554
  👤 MAID terlibat          : 90
  📏 Jarak antar lokasi simultan:
     Min    : 100.3 m
     Median : 196.9 m
     Max    : 10,001.2 m
     Avg    : 0.34 km


---
## 6 · Visualisasi: Distribusi Kecepatan Anomali

In [8]:
# Ambil data kecepatan untuk histogram
speed_data = con.execute(f"""
    SELECT speed_kmph
    FROM read_parquet('{SPEED_ANOMALY_FILE}')
    WHERE speed_kmph <= 100000  -- clip outlier ekstrem untuk visualisasi
""").df()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram distribusi kecepatan anomali
ax1 = axes[0]
ax1.hist(speed_data['speed_kmph'], bins=100, color='#e74c3c', alpha=0.8, edgecolor='white')
ax1.set_xlabel('Kecepatan (km/jam)', fontsize=12)
ax1.set_ylabel('Jumlah Record', fontsize=12)
ax1.set_title('Distribusi Kecepatan Anomali', fontsize=14, fontweight='bold')
ax1.axvline(x=SPEED_THRESHOLD_KMPH, color='black', linestyle='--',
            label=f'Threshold ({SPEED_THRESHOLD_KMPH} km/jam)')
ax1.legend()

# Log-scale histogram
ax2 = axes[1]
ax2.hist(speed_data['speed_kmph'], bins=100, color='#e74c3c', alpha=0.8,
         edgecolor='white', log=True)
ax2.set_xlabel('Kecepatan (km/jam)', fontsize=12)
ax2.set_ylabel('Jumlah Record (log scale)', fontsize=12)
ax2.set_title('Distribusi Kecepatan Anomali (Log Scale)', fontsize=14, fontweight='bold')
ax2.axvline(x=SPEED_THRESHOLD_KMPH, color='black', linestyle='--',
            label=f'Threshold ({SPEED_THRESHOLD_KMPH} km/jam)')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'speed_anomaly_distribution.png'), dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()
print("📊 Plot disimpan ke speed_anomaly_distribution.png")

📊 Plot disimpan ke speed_anomaly_distribution.png


/tmp/ipykernel_168821/3837543105.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 7 · Visualisasi: Distribusi Jarak Bi-Location

In [9]:
# Ambil data jarak bi-location untuk histogram
biloc_data = con.execute(f"""
    SELECT distance_m / 1000.0 AS distance_km
    FROM read_parquet('{BILOC_ANOMALY_FILE}')
""").df()

if len(biloc_data) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(biloc_data['distance_km'], bins=80, color='#8e44ad', alpha=0.8,
            edgecolor='white')
    ax.set_xlabel('Jarak antar lokasi simultan (km)', fontsize=12)
    ax.set_ylabel('Jumlah Pasangan', fontsize=12)
    ax.set_title('Distribusi Jarak Bi-Location Anomali', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'bilocation_distance_distribution.png'), dpi=150,
                bbox_inches='tight', facecolor='white')
    plt.show()
    print("📊 Plot disimpan ke bilocation_distance_distribution.png")
else:
    print("ℹ️  Tidak ada anomali bi-location — tidak ada plot.")

/tmp/ipykernel_168821/1745897715.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


📊 Plot disimpan ke bilocation_distance_distribution.png


---
## 8 · Top MAID dengan Anomali Terbanyak

In [10]:
# Top 20 MAID dengan anomali kecepatan terbanyak
top_speed_maids = con.execute(f"""
    SELECT
        maid,
        COUNT(*)            AS anomaly_count,
        MAX(speed_kmph)     AS max_speed_kmph,
        AVG(speed_kmph)     AS avg_speed_kmph,
        MAX(distance_m)/1000 AS max_distance_km
    FROM read_parquet('{SPEED_ANOMALY_FILE}')
    GROUP BY maid
    ORDER BY anomaly_count DESC
    LIMIT 20
""").df()

print("=" * 70)
print("TOP 20 MAID — ANOMALI KECEPATAN TERBANYAK")
print("=" * 70)
print(top_speed_maids.to_string(index=False))

TOP 20 MAID — ANOMALI KECEPATAN TERBANYAK
                                maid  anomaly_count  max_speed_kmph  avg_speed_kmph  max_distance_km
8516b4c0-d7df-4f73-ad5a-f21039f32d0c            752     5060.049313      886.546689         1.430095
27683c4b-5c14-4a14-9788-9aba0a7b0b38            418     4997.918208      756.549468         1.390074
5afe0024-44d8-458a-9e6f-313a9a21f563            225     4838.975187      852.768187         1.543452
4213fc32-db1f-4322-be8e-e4230784254d            151     5138.167064      846.924332         1.427269
dfe1c884-187c-4269-91ad-cd43c503a814            130      902.124426      282.531008         1.147745
62be6b48-16c9-4468-b035-40a7ce14aa22            128     3462.469716      392.640302         1.049375
7ee98afd-abd0-4429-b311-8145ea57a52a            123     4167.339151      460.185694         2.472420
d60784a3-f519-447b-b90d-fb66c49921b6            105      458.949306      221.704683         0.752403
f9b98635-dc5b-41a8-14f4-8781d4cbb635            1

In [11]:
# Top 20 MAID dengan anomali bi-location terbanyak
top_biloc_maids = con.execute(f"""
    SELECT
        maid,
        COUNT(*)              AS pair_count,
        MAX(distance_m)/1000  AS max_distance_km,
        AVG(distance_m)/1000  AS avg_distance_km
    FROM read_parquet('{BILOC_ANOMALY_FILE}')
    GROUP BY maid
    ORDER BY pair_count DESC
    LIMIT 20
""").df()

print("=" * 70)
print("TOP 20 MAID — ANOMALI BI-LOCATION TERBANYAK")
print("=" * 70)
if len(top_biloc_maids) > 0:
    print(top_biloc_maids.to_string(index=False))
else:
    print("  ℹ️  Tidak ada anomali bi-location ditemukan.")

TOP 20 MAID — ANOMALI BI-LOCATION TERBANYAK
                                maid  pair_count  max_distance_km  avg_distance_km
8516b4c0-d7df-4f73-ad5a-f21039f32d0c         540         1.096172         0.354698
27683c4b-5c14-4a14-9788-9aba0a7b0b38         311         0.727313         0.253792
4213fc32-db1f-4322-be8e-e4230784254d         169         0.677269         0.308184
e0c91767-98b2-4021-9af0-f3e0fe742e2e          78         0.340661         0.291196
5aa409f7-069f-4cc0-90a0-6bd6bcf700bb          50         0.317020         0.153022
41748829-e7e0-42ef-84cc-5bf3a7b67be2          29         0.188937         0.186025
74092e84-2e84-80e5-3542-57e734fd36e2          25         0.223493         0.221172
5f005b3e-ca0b-422f-a643-50f1c34ccebf          21         0.298246         0.241073
5afe0024-44d8-458a-9e6f-313a9a21f563          18         1.347537         0.745198
73bc8164-2f15-4531-ba1a-990af96d031d          17         0.360922         0.359432
25a6fbd2-b6d8-413c-afc4-e466d4f8786a       

---
## 9 · Breakdown Kecepatan per Kategori

In [12]:
# Kategorisasi tier kecepatan anomali
speed_tiers = con.execute(f"""
    SELECT
        CASE
            WHEN speed_kmph BETWEEN 150 AND 300 THEN '150-300 km/jam (kendaraan sangat cepat)'
            WHEN speed_kmph BETWEEN   300 AND    500 THEN '300-500 km/jam (sangat cepat)'
            WHEN speed_kmph BETWEEN   500 AND  1_000 THEN '500-1000 km/jam (pesawat)'
            WHEN speed_kmph BETWEEN 1_000 AND 10_000 THEN '1k-10k km/jam (supersonik)'
            WHEN speed_kmph > 10_000                 THEN '>10k km/jam (teleportasi)'
        END AS kategori,
        COUNT(*)             AS jumlah,
        COUNT(DISTINCT maid) AS maid_terlibat
    FROM read_parquet('{SPEED_ANOMALY_FILE}')
    GROUP BY kategori
    ORDER BY MIN(speed_kmph)
""").df()

print("=" * 70)
print("BREAKDOWN ANOMALI PER KATEGORI KECEPATAN")
print("=" * 70)
print(speed_tiers.to_string(index=False))

BREAKDOWN ANOMALI PER KATEGORI KECEPATAN
                               kategori  jumlah  maid_terlibat
150-300 km/jam (kendaraan sangat cepat)   10316           3336
          300-500 km/jam (sangat cepat)    3249           1514
              500-1000 km/jam (pesawat)    1967            965
             1k-10k km/jam (supersonik)    1363            643
              >10k km/jam (teleportasi)      15              9


---
## 10 · Preview Record Anomali

In [13]:
# Preview 10 anomali kecepatan tercepat
top_speed_records = con.execute(f"""
    SELECT
        maid,
        prev_time,
        curr_time,
        time_diff_s,
        ROUND(prev_lat, 5)  AS prev_lat,
        ROUND(prev_lon, 5)  AS prev_lon,
        ROUND(curr_lat, 5)  AS curr_lat,
        ROUND(curr_lon, 5)  AS curr_lon,
        ROUND(distance_m / 1000, 2)  AS distance_km,
        ROUND(speed_kmph, 1) AS speed_kmph
    FROM read_parquet('{SPEED_ANOMALY_FILE}')
    ORDER BY speed_kmph DESC
    LIMIT 10
""").df()

print("=" * 70)
print("TOP 10 RECORD ANOMALI — KECEPATAN TERTINGGI")
print("=" * 70)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(top_speed_records.to_string(index=False))

TOP 10 RECORD ANOMALI — KECEPATAN TERTINGGI
                                maid                 prev_time                 curr_time  time_diff_s  prev_lat  prev_lon  curr_lat  curr_lon  distance_km  speed_kmph
709c32c4-6f53-46e5-a3f4-eb22f37acf24 2021-10-30 19:56:56+07:00 2021-10-30 19:56:57+07:00            1  -7.74577 110.34281  -7.76250 110.43200        10.00     36004.2
0a095fa4-2a70-41d8-b602-ce66b53d4c93 2022-03-21 14:10:14+07:00 2022-03-21 14:10:15+07:00            1  -7.74877 110.36183  -7.75692 110.41269         5.68     20434.7
0a095fa4-2a70-41d8-b602-ce66b53d4c93 2022-03-21 14:10:12+07:00 2022-03-21 14:10:13+07:00            1  -7.75692 110.41268  -7.74877 110.36183         5.68     20431.4
5e992a31-4530-4e84-98b4-e4bba482dba7 2022-01-13 11:19:05+07:00 2022-01-13 11:19:06+07:00            1  -7.74512 110.34914  -7.75770 110.39861         5.63     20257.6
709c32c4-6f53-46e5-a3f4-eb22f37acf24 2021-10-31 09:47:44+07:00 2021-10-31 09:47:46+07:00            2  -7.76250 110.43200

In [14]:
# Preview 10 anomali bi-location terjauh
top_biloc_records = con.execute(f"""
    SELECT
        maid,
        event_time,
        ROUND(lat_a, 5)  AS lat_a,
        ROUND(lon_a, 5)  AS lon_a,
        ROUND(lat_b, 5)  AS lat_b,
        ROUND(lon_b, 5)  AS lon_b,
        ROUND(distance_m / 1000, 2)  AS distance_km
    FROM read_parquet('{BILOC_ANOMALY_FILE}')
    ORDER BY distance_m DESC
    LIMIT 10
""").df()

print("=" * 70)
print("TOP 10 RECORD ANOMALI — BI-LOCATION TERJAUH")
print("=" * 70)
if len(top_biloc_records) > 0:
    print(top_biloc_records.to_string(index=False))
else:
    print("  ℹ️  Tidak ada anomali bi-location.")

TOP 10 RECORD ANOMALI — BI-LOCATION TERJAUH
                                maid                event_time    lat_a     lon_a    lat_b     lon_b  distance_km
709c32c4-6f53-46e5-a3f4-eb22f37acf24 2021-10-30 17:37:43+07:00 -7.76250 110.43200 -7.74577 110.34281        10.00
cd6c87cf-93fe-4a03-82a0-d95bccc2319a 2021-11-12 20:09:25+07:00 -7.77568 110.43401 -7.74791 110.35915         8.81
cd6c87cf-93fe-4a03-82a0-d95bccc2319a 2021-11-12 20:09:25+07:00 -7.77568 110.43401 -7.74792 110.35916         8.81
45297f31-47ce-4d80-be22-c302c1519cd3 2022-06-01 19:30:20+07:00 -7.76368 110.40618 -7.75750 110.38485         2.45
45297f31-47ce-4d80-be22-c302c1519cd3 2022-06-01 19:30:20+07:00 -7.76368 110.40618 -7.75750 110.38485         2.45
45297f31-47ce-4d80-be22-c302c1519cd3 2022-06-01 19:30:15+07:00 -7.76368 110.40617 -7.75750 110.38486         2.45
5b5713c7-13c7-5956-0584-ff68ecd57bd4 2022-03-06 22:59:02+07:00 -7.76335 110.38903 -7.75029 110.37153         2.41
5b5713c7-13c7-5956-0584-ff68ecd57bd4 2022-03

---
## 11 · Ringkasan

In [15]:
speed_total = con.execute(f"SELECT COUNT(*) FROM read_parquet('{SPEED_ANOMALY_FILE}')").fetchone()[0]
speed_maids = con.execute(f"SELECT COUNT(DISTINCT maid) FROM read_parquet('{SPEED_ANOMALY_FILE}')").fetchone()[0]
biloc_total = con.execute(f"SELECT COUNT(*) FROM read_parquet('{BILOC_ANOMALY_FILE}')").fetchone()[0]
biloc_maids = con.execute(f"SELECT COUNT(DISTINCT maid) FROM read_parquet('{BILOC_ANOMALY_FILE}')").fetchone()[0]

print("=" * 70)
print("              📋 RINGKASAN DETEKSI ANOMALI PERGERAKAN CEPAT")
print("              Data GPS Oktober 2021 – Juni 2022")
print("=" * 70)
print(f"""
  📊 Data Input
     Total record         : {total_rows:,}
     Distinct MAID        : {distinct_maid:,}

  ⚡ Anomali Tipe 1 — Kecepatan Tidak Wajar (>{SPEED_THRESHOLD_KMPH} km/jam)
     Record anomali       : {speed_total:,}
     MAID terlibat        : {speed_maids:,}
     Persentase record    : {speed_total / total_rows * 100:.4f}%
     Persentase MAID      : {speed_maids / distinct_maid * 100:.2f}%
     Output               : {os.path.basename(SPEED_ANOMALY_FILE)}

  🔀 Anomali Tipe 2 — Bi-Location (Lokasi Simultan, ≥{BI_LOCATION_MIN_DIST_M}m)
     Pasangan anomali     : {biloc_total:,}
     MAID terlibat        : {biloc_maids:,}
     Output               : {os.path.basename(BILOC_ANOMALY_FILE)}

  📁 Output disimpan di : {OUTPUT_DIR}/
""")
print("=" * 70)
print("  ✅ Deteksi anomali selesai!")
print("=" * 70)

              📋 RINGKASAN DETEKSI ANOMALI PERGERAKAN CEPAT
              Data GPS Oktober 2021 – Juni 2022

  📊 Data Input
     Total record         : 8,963,571
     Distinct MAID        : 400,643

  ⚡ Anomali Tipe 1 — Kecepatan Tidak Wajar (>150 km/jam)
     Record anomali       : 16,910
     MAID terlibat        : 4,486
     Persentase record    : 0.1887%
     Persentase MAID      : 1.12%
     Output               : speed_anomalies.parquet

  🔀 Anomali Tipe 2 — Bi-Location (Lokasi Simultan, ≥100m)
     Pasangan anomali     : 1,554
     MAID terlibat        : 90
     Output               : bilocation_anomalies.parquet

  📁 Output disimpan di : /home/repal/Github/skripsi/fast-movement-anomali-ring-road/

  ✅ Deteksi anomali selesai!


In [16]:
# Tutup koneksi DuckDB
con.close()
print("✅ Koneksi DuckDB ditutup")

✅ Koneksi DuckDB ditutup
